# Домашнее задание 1. Метод Cross-Entropy, оценка политики и MDP

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IlyaChichkanov/Reinforcement-learning/blob/main/01-intro/homework/homework.ipynb)

**Вес в оценке:** базовое ДЗ, среднее по сложности. Всего 100 баллов + до 30 бонусных.

1. **Практика: метод Cross-Entropy на Frozen Lake 8×8 (40 баллов)**
2. **Практика: оценка политики и ценность состояний (25 баллов)**
3. **Теория: return, марковская цепь, MDP на бумаге (20 баллов)**
4. **Бонус: многорукие бандиты (15 баллов)**
5. **Бонус: DL-разминка (15 баллов)** — для тех, кто прошёл `../seminar/dl_basics.ipynb`

Части с `assert` проверяются автоматически при запуске ячейки: если assert не упал, часть засчитана. Текстовые ответы и графики проверяются вручную. Отвечайте на вопросы прямо в markdown-ячейках.

Первая ячейка — служебная, из лекции: функции для картинок. Дальше — `discounted_return` и `run_session`, тоже из лекции.

In [ ]:
# Если ноутбук открыт в Google Colab: ставим недостающие пакеты. Локально (после uv sync) ячейка ничего не делает.
import importlib.util, subprocess, sys
if importlib.util.find_spec("gymnasium") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gymnasium[toy-text,classic-control]"], check=True)

In [ ]:
# Служебный код для картинок: запустить и не читать. Содержательный код начинается со следующей ячейки.
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from matplotlib import animation
from IPython.display import HTML, display

ARROWS = "←↓→↑"                      # так пронумерованы действия в Frozen Lake: 0=←, 1=↓, 2=→, 3=↑
CELL_COLORS = {"S": "#f2f2f2", "F": "#ffffff", "H": "#a8c8ec", "G": "#a9dfa9"}

def lake_desc(env):
    """Карта озера как список строк: ['SFFF', 'FHFH', 'FFFH', 'HFFG']."""
    return ["".join(c.decode() for c in row) for row in env.unwrapped.desc]

def draw_lake(desc, ax=None, size=3.0):
    """Пустая карта озера. Возвращает оси, поверх которых можно рисовать маршруты и стрелки."""
    nrow, ncol = len(desc), len(desc[0])
    if ax is None:
        _, ax = plt.subplots(figsize=(size, size * nrow / ncol))
    ax.set_xlim(0, ncol); ax.set_ylim(nrow, 0); ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect("equal")
    for r in range(nrow):
        for c in range(ncol):
            ax.add_patch(plt.Rectangle((c, r), 1, 1, facecolor=CELL_COLORS[desc[r][c]], edgecolor="k", lw=0.6))
            if desc[r][c] in "SHG":
                ax.text(c + 0.5, r + 0.82, desc[r][c], ha="center", va="center", fontsize=8, color="#666")
    return ax

def draw_paths(sessions, desc, ax=None, title="", color="C0", alpha=0.35, size=3.0):
    """Маршруты эпизодов поверх карты: ломаная через центры клеток, точка — где эпизод закончился."""
    ax = draw_lake(desc, ax, size)
    ncol, rng = len(desc[0]), np.random.default_rng(0)
    for states, *_ in sessions:
        pts = np.array([divmod(int(s), ncol) for s in states], float) + 0.5 + rng.normal(0, 0.07, (len(states), 2))
        ax.plot(pts[:, 1], pts[:, 0], color=color, alpha=alpha, lw=1.5)
        ax.plot(pts[-1, 1], pts[-1, 0], "o", color=color, alpha=alpha, ms=4)
    ax.set_title(title, fontsize=10)
    return ax

def draw_policy(policy, desc, ax=None, title="", size=3.0):
    """Стрелка — самое вероятное действие, насыщенность — его вероятность; «·» — строка почти равномерная."""
    ax = draw_lake(desc, ax, size)
    ncol = len(desc[0])
    for s, p in enumerate(policy):
        r, c = divmod(s, ncol)
        if desc[r][c] in "HG":
            continue
        a, top2 = int(np.argmax(p)), np.sort(p)[-2:]
        if top2[1] - top2[0] < 0.05:
            ax.text(c + 0.5, r + 0.5, "·", ha="center", va="center", fontsize=18, color="grey")
        else:
            ax.text(c + 0.5, r + 0.5, ARROWS[a], ha="center", va="center", fontsize=20, alpha=0.2 + 0.8 * float(p[a]))
    ax.set_title(title, fontsize=10)
    return ax

def plot_values(values, desc, ax=None, title="", vmax=None, size=3.0):
    """Тепловая карта ценности клеток (проруби и цель — терминальные, у них ценность 0 по определению)."""
    nrow, ncol = len(desc), len(desc[0])
    ax = draw_lake(desc, ax, size)
    grid = np.asarray(values, float).reshape(nrow, ncol)
    vmax = vmax or max(float(np.nanmax(grid)), 1e-9)
    for r in range(nrow):
        for c in range(ncol):
            if desc[r][c] in "HG":
                continue
            if np.isnan(grid[r, c]):                     # клетку ни разу не посещали — оценки нет
                ax.text(c + 0.5, r + 0.5, "—", ha="center", va="center", fontsize=9, color="grey")
                continue
            ax.add_patch(plt.Rectangle((c, r), 1, 1, facecolor=plt.cm.YlOrRd(0.85 * grid[r, c] / vmax), edgecolor="k", lw=0.6))
            ax.text(c + 0.5, r + 0.5, f"{grid[r, c]:.2f}", ha="center", va="center", fontsize=9)
    ax.set_title(title, fontsize=10)
    return ax

def plot_returns(returns, threshold=None, elite_mask=None, ax=None, title=""):
    """Каждая точка — эпизод, по вертикали его return (как на схеме метода Cross-Entropy)."""
    if ax is None:
        _, ax = plt.subplots(figsize=(4.6, 3.2))
    returns, x = np.asarray(returns, float), np.random.default_rng(0).uniform(0, 1, len(returns))
    if elite_mask is None:
        ax.scatter(x, returns, s=22, color="C0", alpha=0.7)
    else:
        m = np.asarray(elite_mask, bool)
        ax.scatter(x[~m], returns[~m], s=18, color="grey", alpha=0.35, label=f"остальные ({(~m).sum()})")
        ax.scatter(x[m], returns[m], s=42, color="C2", alpha=0.95, edgecolors="white", label=f"элита ({m.sum()})")
        ax.legend(loc="upper right", fontsize=8)
    if threshold is not None:
        ax.axhline(threshold, ls="--", color="C3", lw=1.5)
        ax.text(0.0, threshold + 0.02, f"порог = {threshold:.2f}", color="C3", fontsize=8)
    ax.set_xticks([]); ax.set_ylabel("return"); ax.set_ylim(-0.05, 1.05); ax.set_title(title, fontsize=10)
    return ax

def animate_policy(policies, rates, desc, interval=500):
    """Кадр — одна итерация обучения: стрелки политики и доля успешных эпизодов."""
    nrow, ncol = len(desc), len(desc[0])
    fig, ax = plt.subplots(figsize=(3.4, 3.4 * nrow / ncol + 0.3)); plt.close(fig)
    def frame(k):
        ax.clear(); draw_policy(policies[k], desc, ax=ax, title=f"итерация {k}: доля успехов {rates[k]:.0%}")
    anim = animation.FuncAnimation(fig, frame, frames=len(policies), interval=interval)
    return HTML(anim.to_jshtml(default_mode="loop"))

def show_frames(frames, title="", interval=60, width=3.2):
    """Кадры среды -> анимация в ноутбуке (в превью на GitHub не видна, только при запуске)."""
    h, w = frames[0].shape[:2]
    fig, ax = plt.subplots(figsize=(width, width * h / w))
    ax.axis("off"); ax.set_title(title, fontsize=10)
    im = ax.imshow(frames[0]); plt.close(fig)
    anim = animation.FuncAnimation(fig, lambda i: (im.set_data(frames[i]),), frames=len(frames), interval=interval, blit=True)
    return HTML(anim.to_jshtml(default_mode="loop"))

In [ ]:
GAMMA = 0.95

def discounted_return(rewards, gamma=GAMMA):
    """G = r_0 + γ r_1 + γ² r_2 + ..."""
    return sum(gamma ** t * r for t, r in enumerate(rewards))

def run_session(env, policy, rng, max_steps=100):
    """Один эпизод политикой-таблицей policy. Возвращает состояния (включая последнее), действия и награды."""
    s, _ = env.reset(seed=int(rng.integers(1_000_000)))
    states, actions, rewards = [s], [], []
    for _ in range(max_steps):
        a = int(rng.choice(len(policy[s]), p=policy[s]))      # действие ~ π(· | s)
        s, r, terminated, truncated, _ = env.step(a)
        states.append(s); actions.append(a); rewards.append(r)
        if terminated or truncated:
            break
    return states, actions, rewards


def success_rate(env, policy, n_episodes=500, seed=0):
    """Доля эпизодов, дошедших до цели."""
    rng = np.random.default_rng(seed)
    return np.mean([sum(run_session(env, policy, rng)[2]) > 0 for _ in range(n_episodes)])

env8 = gym.make("FrozenLake-v1", map_name="8x8", is_slippery=False)
desc8 = lake_desc(env8)
n_states, n_actions = env8.observation_space.n, env8.action_space.n
draw_lake(desc8, size=4); plt.show()
print("\n".join(desc8))

## Часть 1. Метод Cross-Entropy на Frozen Lake 8×8 (40 баллов)

На лекции и семинаре мы обучили табличный Cross-Entropy на озере 4×4. Карта 8×8 сложнее: 64 состояния, до цели далеко, случайная политика доходит крайне редко (меньше чем в одном эпизоде из пятисот).

### 1.1 Реализация и обучение (15 баллов)

Соберите алгоритм из трёх функций семинара: `select_elite(sessions, scores, q)`, `update_policy(policy, elite, laplace, mix)` и цикл `cross_entropy_method(env, n_iter, n_sessions, q, laplace, mix, seed)`, возвращающий политику и `log = {"policies": [...], "success": [...], "return": [...]}`. Можно скопировать своё решение с семинара или написать заново.

Обучите агента на `env8`, постройте кривую обучения (доля успехов и средний return по итерациям) и нарисуйте выученную политику. Самопроверка требует **долю успехов ≥ 90%** у выученной стохастической политики. Подсказка: успешных эпизодов у случайной политики очень мало, поэтому сессий нужно несколько сотен на итерацию; помогает сглаживание.

In [ ]:
def select_elite(sessions, scores, q=0.7):
    # TODO: ваш код здесь
    raise NotImplementedError

def update_policy(policy, elite, laplace=0.0, mix=1.0):
    # TODO: ваш код здесь
    raise NotImplementedError

def cross_entropy_method(env, n_iter=15, n_sessions=200, q=0.7, laplace=0.0, mix=1.0, seed=0):
    # TODO: ваш код здесь
    raise NotImplementedError

In [ ]:
# TODO: подберите n_iter, n_sessions, q, laplace, mix
policy8, log8 = cross_entropy_method(env8, n_iter=..., n_sessions=..., q=..., laplace=..., mix=..., seed=0)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
axes[0].plot(log8["success"], marker="."); axes[0].set_xlabel("итерация"); axes[0].set_ylabel("доля успешных эпизодов")
axes[1].plot(log8["return"], marker=".", color="C1"); axes[1].set_xlabel("итерация"); axes[1].set_ylabel("средний return")
draw_policy(policy8, desc8, ax=axes[2], title="выученная политика"); plt.tight_layout(); plt.show()

rate8 = success_rate(env8, policy8, n_episodes=1000)
print(f"доля успехов: {rate8:.1%}")
assert policy8.shape == (64, 4) and np.allclose(policy8.sum(axis=1), 1)
assert rate8 >= 0.9, "политика должна доходить до цели не реже, чем в 90% эпизодов"

### 1.2 Сглаживание (10 баллов)

Сравните три варианта обновления политики: **без сглаживания** (`laplace=0, mix=1`), **только Лаплас** (`laplace=0.5, mix=1`), **Лаплас и смешивание** (`laplace=0.5, mix=0.5`). Для каждого варианта сделайте 3 запуска с разными сидами и постройте кривые доли успехов (все 9 на одном графике, цвет — вариант). Ответьте текстом: какой вариант быстрее, какой стабильнее, и у какого чаще встречаются запуски, «зависшие» на нуле?

In [ ]:
# TODO: ваш код здесь (9 запусков, один график)

_Ваш ответ:_ TODO

### 1.3 Скользкий лёд (10 баллов)

Вернёмся на озеро 4×4, но включим скольжение: `gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)`. Запустите `cross_entropy_method` с параметрами из 1.1 и постройте кривую. Затем:

1. Объясните текстом, почему результат намного хуже, чем на гладком льду: что именно попадает в элиту и почему это плохо для шага 3?
2. Предложите **не меньше двух** изменений (например: больше эпизодов на итерацию, выше `q`, меньше `mix`, больше итераций) и проверьте каждое на графике. Цель — доля успехов ≥ 35% (лучшая детерминированная таблица даёт около 74%; табличный CEM до неё не дотягивает, и это нормально — объясните, почему).

In [ ]:
slip = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)
# TODO: ваш код здесь

_Ваш ответ:_ TODO

### 1.4 Два вопроса про выученную политику (5 баллов)

1. Стрелки на картинке политики показывают самое вероятное действие. Почему **жадная** версия выученной политики (`argmax` в каждой клетке) иногда не доходит до цели, хотя стохастическая доходит почти всегда? Подсказка: посмотрите на клетки с точкой.
2. Почему нулевая вероятность действия опаснее, чем просто маленькая? Свяжите ответ с разделом «исследование и использование» лекции.

_Ваш ответ:_ TODO

## Часть 2. Оценка политики и ценность состояний (25 баллов)

### 2.1 Три политики (10 баллов)

Оцените методом Монте-Карло три политики на `env8`: равномерную, **маршрут, написанный руками** (словарь «клетка → действие», как на семинаре; подсказка: вдоль верхнего края, потом вдоль правого), и выученную в 1.1. Для каждой посчитайте средний return и его стандартную ошибку по 1000 эпизодам и нарисуйте столбчатую диаграмму с ошибками. Ответьте текстом: сколько эпизодов нужно, чтобы стандартная ошибка оценки равномерной политики стала меньше 0.001?

In [ ]:
LEFT, DOWN, RIGHT, UP = 0, 1, 2, 3

def table_policy(route, n_states=64, n_actions=4, default=LEFT):
    policy = np.zeros((n_states, n_actions))
    for s in range(n_states):
        policy[s, route.get(s, default)] = 1.0
    return policy

def evaluate(env, policy, n_episodes=1000, seed=0):
    # TODO: вернуть (средний return, стандартная ошибка)
    raise NotImplementedError

route8 = {}   # TODO: маршрут руками
uniform8 = np.ones((64, 4)) / 4
# TODO: оценка трёх политик и диаграмма

_Ваш ответ:_ TODO

### 2.2 Тепловая карта ценности (10 баллов)

Возьмите `estimate_values(sessions, n_states)` с семинара (ценность клетки — средний return от первого попадания в неё) и постройте тепловую карту $V^\pi$ для выученной политики из 1.1 по 3000 эпизодам (`plot_values` есть в служебной ячейке). Ответьте текстом: почему ценность растёт к цели и чему равен её максимум; что означают прочерки; почему у прорубей ценности нет.

In [ ]:
def estimate_values(sessions, n_states, gamma=GAMMA):
    # TODO: ваш код здесь
    raise NotImplementedError

# TODO: тепловая карта V для policy8

_Ваш ответ:_ TODO

### 2.3 Ценность действия (5 баллов)

Выберите клетку на пути выученной политики, но не соседнюю с целью. По тем же эпизодам оцените $Q^\pi(s, a)$ для всех четырёх действий (усредняйте return от первого попадания в клетку **при условии**, что в ней сделали действие $a$; если какое-то действие в этой клетке ни разу не встретилось, так и скажите). Совпадает ли лучшее действие с тем, что показывает стрелка политики?

In [ ]:
# TODO: ваш код здесь

_Ваш ответ:_ TODO

## Часть 3. Теория (20 баллов)

Отвечайте прямо в markdown-ячейках, используя LaTeX ($...$ или $$...$$). Показывайте промежуточные шаги, а не только финальный ответ.

### 3.1 Return для простого эпизода (5 баллов)

Дан эпизод (последовательность наград после каждого шага):

$$
R_0 = 1,\; R_1 = 0,\; R_2 = 2,\; R_3 = 0,\; R_4 = 1 \quad \text{(эпизод завершается после шага 4)}
$$

Посчитайте $G_0, G_1, \ldots, G_4$ для $\gamma = 0.9$. Покажите, как использовать рекуррентное соотношение $G_t = R_t + \gamma G_{t+1}$, чтобы не считать каждую сумму с нуля. Проверьте $G_0$ функцией `discounted_return`.

_Ваш ответ:_ TODO

### 3.2 Марковская цепь (5 баллов)

Погода в городе описывается марковской цепью с состояниями {Солнце, Облачно, Дождь} и матрицей переходов

$$
P = \begin{pmatrix} 0.7 & 0.2 & 0.1 \\ 0.3 & 0.4 & 0.3 \\ 0.2 & 0.3 & 0.5 \end{pmatrix}.
$$

1. Сегодня солнце. Какова вероятность дождя послезавтра? Запишите вычисление через $p_0 P^2$.
2. Найдите стационарное распределение $\pi^\top P = \pi^\top$ (аналитически или численно в ячейке ниже — но запишите систему уравнений).
3. Проверьте численно: возведите $P$ в большую степень и сравните строки с найденным $\pi$.

_Ваш ответ:_ TODO

In [ ]:
P_weather = np.array([[0.7, 0.2, 0.1],
                      [0.3, 0.4, 0.3],
                      [0.2, 0.3, 0.5]])
# TODO: p_0 @ P^2, стационарное распределение, проверка через np.linalg.matrix_power

### 3.3 MDP на бумаге: робот-уборщик (5 баллов)

Робот-уборщик находится в одной из трёх комнат: `{Кухня, Зал, Коридор}`. Действия: `{Убирать, Переехать}`.

* Из **Кухни**: `Убирать` -> остаться в Кухне с наградой +2 (вероятность 1); `Переехать` -> Зал с наградой 0 (вероятность 0.8) или Коридор с наградой 0 (вероятность 0.2).
* Из **Зала**: `Убирать` -> остаться в Зале с наградой +1 (вероятность 1); `Переехать` -> Кухня с наградой 0 (вероятность 0.5) или Коридор с наградой 0 (вероятность 0.5).
* Из **Коридора**: `Убирать` -> остаться в Коридоре с наградой 0 (в коридоре убирать нечего); `Переехать` -> Кухня с наградой 0 (вероятность 0.5) или Зал с наградой 0 (вероятность 0.5).

1. Выпишите MDP формально: множества $\mathcal{S}$ и $\mathcal{A}$, таблицу $P(s' \mid s, a)$ (два массива 3×3 — по одному на действие) и $R(s, a)$. Нарисуйте граф переходов (можно словами: «стрелка из … в … с вероятностью …»).
2. Пусть политика $\pi$ детерминированная: `Убирать` в Кухне и Зале, `Переехать` в Коридоре. Запишите матрицу переходов марковской цепи, которая получается при фиксированной политике, и ожидаемую награду за один шаг из каждого состояния.
3. Является ли процесс марковским, если робот видит только номер комнаты, но награда за уборку зависит от того, сколько шагов назад в ней убирали? Как исправить состояние?

_Ваш ответ:_ TODO

In [ ]:
# Опционально: запишите P_pi и R_pi численно и проверьте, что строки P_pi суммируются в единицу.
# P_pi = np.array(...); R_pi = np.array(...)

### 3.4 Сформулируйте как MDP (5 баллов)

Для **двух** задач из списка опишите MDP: множество состояний (что агент знает, чего не знает), действия, переходы и **где в них случайность**, награда за шаг, эпизод (или задача непрерывная).

* Лифт в 10-этажном доме, который решает, куда ехать.
* Игра «2048».
* Полив теплицы: датчики влажности и температуры, насос.
* Любая задача из вашей жизни.

_Ваш ответ:_ TODO

## Часть 4. Бонус: многорукие бандиты (15 баллов)

**Многорукий бандит** — MDP с одним состоянием: есть $K$ игровых автоматов («рук»), у каждой своя неизвестная вероятность выигрыша, на каждом шаге вы дёргаете одну руку и получаете 0 или 1. Никаких переходов между состояниями нет, зато дилемма исследования и использования — в чистом виде. **Ценность действия** здесь — это просто средний выигрыш руки, $Q(a) = \mathbb{E}[R \mid A = a]$: та же величина, что в 2.3, только без состояний.

Среда дана. Реализуйте агента **ε-greedy**: с вероятностью ε выбирает случайную руку, иначе — руку с наибольшей оценкой $\hat Q(a)$; оценка обновляется инкрементально после каждого шага, $\hat Q(a) \leftarrow \hat Q(a) + \frac{1}{N(a)}\big(r - \hat Q(a)\big)$ (это то же самое, что среднее по всем наградам руки). Постройте кривые накопленного **regret** — $\sum_t (p^* - p_{A_t})$, разницы между лучшей рукой и выбранной, — для ε ∈ {0, 0.01, 0.1}, усреднив по 20 запускам. Объясните форму кривых: почему у ε = 0 regret иногда растёт быстрее всех, а у ε = 0.1 — линейно даже после того, как лучшая рука найдена.

**+5 баллов**: реализуйте UCB1 ($A_t = \arg\max_a \hat Q(a) + c\sqrt{\ln t / N(a)}$) и добавьте на график.

In [ ]:
class BernoulliBanditEnv:
    def __init__(self, probs, rng):
        self.probs, self.rng = np.array(probs, dtype=float), rng
    def pull(self, arm):
        return float(self.rng.random() < self.probs[arm])

class EpsilonGreedyAgent:
    def __init__(self, n_arms, epsilon, rng):
        self.epsilon, self.rng = epsilon, rng
        self.Q, self.N = np.zeros(n_arms), np.zeros(n_arms)
    def select_arm(self):
        # TODO: ваш код здесь
        raise NotImplementedError
    def update(self, arm, reward):
        # TODO: ваш код здесь
        raise NotImplementedError

def run_bandit(make_agent, probs=(0.1, 0.5, 0.3, 0.55, 0.45), n_steps=2000, n_runs=20):
    # средний накопленный regret по n_runs запускам
    regrets = []
    for run in range(n_runs):
        rng = np.random.default_rng(run)
        env, agent = BernoulliBanditEnv(probs, rng), make_agent(len(probs), rng)
        regret, total = [], 0.0
        for t in range(n_steps):
            arm = agent.select_arm(); r = env.pull(arm); agent.update(arm, r)
            total += env.probs.max() - env.probs[arm]; regret.append(total)
        regrets.append(regret)
    return np.mean(regrets, axis=0)

# TODO: кривые regret для ε ∈ {0, 0.01, 0.1}

_Ваш ответ:_ TODO

## Часть 5. Бонус: DL-разминка (15 баллов)

Три упражнения из `../seminar/dl_basics.ipynb`, каждое по 5 баллов. Нужен PyTorch (`pip install torch`, хватит CPU-версии).

### 5.1 Градиент руками и autograd (5 баллов)

Для линейной модели $\hat y = w x + b$ и функции потерь $L = \frac{1}{n}\sum_i (\hat y_i - y_i)^2$ выпишите $\partial L / \partial w$ и $\partial L / \partial b$ и посчитайте их в numpy. Сравните с тем, что даёт `torch.autograd`.

In [ ]:
import torch

rng = np.random.default_rng(0)
x = rng.uniform(-1, 1, 50); y = 2 * x - 1 + rng.normal(0, 0.1, 50)
w0, b0 = 0.5, 0.0

def grad_manual(w, b, x, y):
    # TODO: вернуть (dL/dw, dL/db) по формулам
    raise NotImplementedError

w = torch.tensor(w0, requires_grad=True); b = torch.tensor(b0, requires_grad=True)
loss = ((w * torch.tensor(x) + b - torch.tensor(y)) ** 2).mean()
loss.backward()
dw, db = grad_manual(w0, b0, x, y)
assert np.isclose(dw, w.grad.item(), atol=1e-6) and np.isclose(db, b.grad.item(), atol=1e-6)
print("ok: градиенты совпали", round(dw, 4), round(db, 4))

### 5.2 Регрессия нейросетью (5 баллов)

Обучите небольшую сеть (`nn.Sequential` из двух скрытых слоёв по 32 нейрона с `Tanh`) приближать $f(x) = \sin 3x$ на отрезке $[-1, 1]$. Цель — среднеквадратичная ошибка на сетке из 200 точек меньше 0.01. Нарисуйте функцию и её приближение.

In [ ]:
import torch.nn as nn

xs = torch.linspace(-1, 1, 200).unsqueeze(1)
ys = torch.sin(3 * xs)

# TODO: модель, оптимизатор, цикл обучения
model = ...

with torch.no_grad():
    mse = ((model(xs) - ys) ** 2).mean().item()
plt.plot(xs, ys, label="sin 3x"); plt.plot(xs, model(xs).detach(), "--", label="сеть"); plt.legend(); plt.show()
print(f"MSE = {mse:.4f}")
assert mse < 0.01

### 5.3 Классификация: две спирали (5 баллов)

Данные — две перепутанные спирали (генератор ниже). Обучите классификатор (`nn.Sequential`, на выходе два логита, функция потерь `nn.CrossEntropyLoss` — та же кросс-энтропия, что в методе Cross-Entropy из лекции) и добейтесь точности не ниже 95% на отложенной выборке. Нарисуйте точки и раскраску плоскости предсказаниями сети.

In [ ]:
def make_spirals(n=300, noise=0.15, seed=0):
    rng = np.random.default_rng(seed)
    t = np.linspace(0.4, 3.0, n)
    X, y = [], []
    for k in range(2):
        angle = t * np.pi + k * np.pi
        X.append(np.stack([t * np.cos(angle), t * np.sin(angle)], 1) + rng.normal(0, noise, (n, 2))); y.append(np.full(n, k))
    X, y = np.concatenate(X).astype(np.float32), np.concatenate(y)
    idx = rng.permutation(len(y))
    return X[idx], y[idx]

X, y = make_spirals()
X_train, y_train, X_test, y_test = torch.tensor(X[:400]), torch.tensor(y[:400]), torch.tensor(X[400:]), torch.tensor(y[400:])

# TODO: модель, обучение
clf = ...

with torch.no_grad():
    acc = (clf(X_test).argmax(1) == y_test).float().mean().item()
print(f"точность на отложенной выборке: {acc:.1%}")
assert acc >= 0.95

## Чек-лист перед сдачей

- [ ] 1.1: `cross_entropy_method` проходит self-check на 8×8 (доля успехов ≥ 90%), есть кривые и картинка политики
- [ ] 1.2: 9 кривых на одном графике и текстовый вывод про сглаживание
- [ ] 1.3: кривые на скользком льду, объяснение и не меньше двух проверенных изменений
- [ ] 1.4: ответы на два вопроса
- [ ] 2.1: три политики оценены с ошибками, диаграмма, ответ про число эпизодов
- [ ] 2.2: тепловая карта ценности и её объяснение
- [ ] 2.3: $Q$ для четырёх действий в выбранной клетке
- [ ] 3.1–3.4: return, марковская цепь, MDP робота-уборщика, две задачи как MDP
- [ ] (бонус) 4: ε-greedy и кривые regret; UCB1
- [ ] (бонус) 5: три упражнения DL-разминки проходят assert